In [ ]:
"""Elasticsearch Practical 2 - Employee CRUD using the REST API via ``requests``.

Tasks implemented (in order):
 1. Create an index called ``employees``.
 2. Insert documents using the Index API (explicit IDs).
 3. Insert documents with automatically generated IDs.
 4. Retrieve documents using their IDs.
 5. Update individual document fields (full replace).
 6. Perform partial document updates (``_update`` API).
 7. Delete individual documents.
 8. Delete the entire index.
 9. Recreate the index.
10. Insert multiple documents using the Bulk API.

Run the generator first so ``employees.json`` exists:

    python generate_employees.py
    python main.py
"""

import json
import os

import requests

BASE_URL = os.getenv("ES_URL", "http://localhost:9200").rstrip("/")
INDEX = "employees"
HERE = os.path.dirname(os.path.abspath(__file__))
DATA_FILE = os.path.join(HERE, "employees.json")

INDEX_BODY = {
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {
        "properties": {
            "employee_id": {"type": "integer"},
            "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
            "department": {"type": "keyword"},
            "designation": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
            "salary": {"type": "integer"},
            "city": {"type": "keyword"},
            "joining_date": {"type": "date"},
            "active": {"type": "boolean"},
        }
    },
}




In [ ]:
def es(method, path, body=None, ndjson=None):
    """Perform a request against Elasticsearch and return parsed JSON."""
    headers = {}
    kwargs = {}
    if ndjson is not None:
        headers["Content-Type"] = "application/x-ndjson"
        kwargs["data"] = ndjson
    elif body is not None:
        headers["Content-Type"] = "application/json"
        kwargs["data"] = json.dumps(body)

    resp = requests.request(method, BASE_URL + path, headers=headers, timeout=30, **kwargs)
    if resp.status_code >= 400:
        try:
            detail = resp.json()
        except ValueError:
            detail = resp.text
        raise RuntimeError(f"{method} {path} -> {resp.status_code}: {detail}")
    if not resp.content:
        return None
    return resp.json()


In [ ]:
def index_exists():
    return requests.head(f"{BASE_URL}/{INDEX}", timeout=30).status_code == 200

In [ ]:
def wait_for_es():
    try:
        requests.get(BASE_URL, timeout=5).raise_for_status()
        return True
    except requests.RequestException as exc:
        print(f"[!] Cannot reach Elasticsearch at {BASE_URL}: {exc}")
        print("    Start it first:  docker compose up -d   (in Practical_1)")
        return False


In [ ]:
def load_dataset():
    with open(DATA_FILE, encoding="utf-8") as handle:
        return json.load(handle)


In [ ]:
def banner(text):
    print("\n" + "=" * 70)
    print(text)
    print("=" * 70)

In [ ]:
def drop_index_if_exists():
    if index_exists():
        es("DELETE", f"/{INDEX}")
        print(f"Dropped existing '{INDEX}' index.")

In [ ]:
def task1_create_index():
    banner("Task 1 - Create index 'employees'")
    resp = es("PUT", f"/{INDEX}", body=INDEX_BODY)
    print(f"acknowledged={resp['acknowledged']} shards_acknowledged={resp['shards_acknowledged']}")


In [ ]:
def task2_index_api(docs):
    banner("Task 2 - Insert documents using the Index API (explicit IDs)")
    ids = []
    for doc in docs[:5]:
        emp_id = doc["employee_id"]
        resp = es("POST", f"/{INDEX}/_doc/{emp_id}", body=doc)
        ids.append(emp_id)
        print(f"indexed employee_id={emp_id} -> _id={resp['_id']} result={resp['result']}")
    return ids

In [ ]:
def task3_auto_ids(docs):
    banner("Task 3 - Insert documents with auto-generated IDs")
    auto_ids = []
    for doc in docs[5:8]:
        resp = es("POST", f"/{INDEX}/_doc", body=doc)
        auto_ids.append(resp["_id"])
        print(f"indexed {doc['name']} -> auto _id={resp['_id']} result={resp['result']}")
    return auto_ids

In [ ]:
def task4_get_by_id(explicit_id, auto_id):
    banner("Task 4 - Retrieve documents using their IDs")
    for doc_id in (explicit_id, auto_id):
        resp = es("GET", f"/{INDEX}/_doc/{doc_id}")
        src = resp["_source"]
        print(f"_id={resp['_id']} found={resp['found']} name={src['name']} designation={src['designation']}")


In [ ]:
def task5_full_replace(emp_id, docs):
    banner("Task 5 - Update individual fields (full replace with same _id)")
    original = next(d for d in docs if d["employee_id"] == emp_id)
    updated = dict(original)
    old_salary = updated["salary"]
    updated["salary"] = old_salary + 5000
    if not updated["designation"].startswith("Senior"):
        updated["designation"] = "Senior " + updated["designation"]
    updated["city"] = "Bengaluru"

    resp = es("PUT", f"/{INDEX}/_doc/{emp_id}", body=updated)
    print(f"replaced _id={emp_id} result={resp['result']} version={resp['_version']}")
    print(f"  salary {old_salary} -> {updated['salary']}, designation -> {updated['designation']}")


In [ ]:
def task6_partial_update(emp_id):
    banner("Task 6 - Partial document update (_update API)")
    current = es("GET", f"/{INDEX}/_doc/{emp_id}")["_source"]
    new_salary = int(current["salary"] * 1.10)
    patch = {"salary": new_salary, "active": True}

    resp = es("POST", f"/{INDEX}/_update/{emp_id}", body={"doc": patch})
    print(f"partial update _id={emp_id} result={resp['result']} version={resp['_version']}")
    print(f"  changed only: {patch}")

In [ ]:

def task7_delete_document(emp_id):
    banner("Task 7 - Delete an individual document")
    resp = es("DELETE", f"/{INDEX}/_doc/{emp_id}")
    print(f"deleted _id={emp_id} result={resp['result']}")


In [ ]:
def task8_delete_index():
    banner("Task 8 - Delete the entire index")
    resp = es("DELETE", f"/{INDEX}")
    print(f"acknowledged={resp['acknowledged']}")

In [ ]:

def task9_recreate_index():
    banner("Task 9 - Recreate the index")
    resp = es("PUT", f"/{INDEX}", body=INDEX_BODY)
    print(f"acknowledged={resp['acknowledged']} shards_acknowledged={resp['shards_acknowledged']}")


In [ ]:
def task10_bulk_index(docs):
    banner("Task 10 - Insert multiple documents using the Bulk API")
    lines = []
    for doc in docs:
        action = {"index": {"_index": INDEX, "_id": str(doc["employee_id"])}}
        lines.append(json.dumps(action))
        lines.append(json.dumps(doc))
    payload = "\n".join(lines) + "\n"

    resp = es("POST", "/_bulk", ndjson=payload)
    failed = sum(1 for item in resp["items"] if item["index"].get("error"))
    print(f"bulk indexed={len(resp['items'])} errors={resp['errors']} failed={failed}")

    es("POST", f"/{INDEX}/_refresh")
    count = es("GET", f"/{INDEX}/_count")["count"]
    print(f"documents now in '{INDEX}': {count}")


In [ ]:
def main():
    if not wait_for_es():
        raise SystemExit(1)

    docs = load_dataset()
    print(f"Loaded {len(docs)} records from {DATA_FILE}")

    drop_index_if_exists()

    task1_create_index()
    explicit_ids = task2_index_api(docs)
    auto_ids = task3_auto_ids(docs)
    task4_get_by_id(explicit_ids[0], auto_ids[0])
    task5_full_replace(explicit_ids[0], docs)
    task6_partial_update(explicit_ids[1])
    task7_delete_document(explicit_ids[-1])
    task8_delete_index()
    task9_recreate_index()
    task10_bulk_index(docs)

    banner("Done - all 10 tasks executed successfully")


if __name__ == "__main__":
    main()
